# HIT401 Simple Scam Email Project — Part 1

## Goal
Prepare a small, understandable and source-separated dataset for scam-email detection.

**Research question:** How well does a model trained on one set of email sources recognise scam emails from different sources?

We use public secondary data downloaded from the Kaggle *Phishing Email Dataset*. We do not claim that we collected these emails ourselves.

## Experiment design

| Purpose | Scam source | Legitimate source |
|---|---|---|
| Training | Nazario | Enron |
| External testing | Nigerian Fraud | SpamAssassin |

This is a **source-separated test**: the test sources are not used for training. CEAS-08 is retained for a later baseline comparison, but it is not used in this main preparation step because its positive class is mainly spam rather than strictly scam.

In [ ]:
# Cell 1: Import the tools used in this notebook
from pathlib import Path
import zipfile
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)
print('Tools loaded successfully.')

## Step 1 — Upload the data ZIP
A dataset is a table. Each **row** is one email, each **column** stores one type of information, and the **label** is the known answer.

In [ ]:
# Cell 2: Upload capstone_data.zip when it is not already in Colab
zip_path = Path('/content/capstone_data.zip')

if not zip_path.exists():
    from google.colab import files
    print('Choose capstone_data.zip from your computer.')
    uploaded = files.upload()
    if 'capstone_data.zip' not in uploaded:
        raise FileNotFoundError('Please upload the file named capstone_data.zip.')

data_root = Path('/content/capstone_project')
with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(data_root)

data_dir = data_root / 'data'
print('Data folder:', data_dir)
print('CSV files:', sorted(p.name for p in data_dir.glob('*.csv')))

## Step 2 — Load the five relevant files
`pandas.read_csv()` opens each CSV file as a DataFrame, which is Python's name for a table.

In [ ]:
# Cell 3: Read the source datasets
files_to_load = {
    'CEAS_08': 'CEAS_08.csv',
    'Nazario': 'Nazario.csv',
    'Nigerian_Fraud': 'Nigerian_Fraud.csv',
    'Enron': 'Enron.csv',
    'SpamAssassin': 'SpamAssasin.csv',  # This is how the supplied file is spelled.
}

raw = {name: pd.read_csv(data_dir / filename) for name, filename in files_to_load.items()}

summary_rows = []
for name, table in raw.items():
    summary_rows.append({
        'source': name,
        'rows': len(table),
        'columns': len(table.columns),
        'label_0': int((table['label'] == 0).sum()),
        'label_1': int((table['label'] == 1).sum()),
    })

source_summary = pd.DataFrame(summary_rows)
source_summary

### How to explain the output
- `rows` is the number of emails.
- `label_0` is the number labelled legitimate.
- `label_1` is the number labelled spam, phishing or fraud according to that source.
- Labels do not have exactly the same meaning in every corpus, so we keep the source name.

In [ ]:
# Cell 4: Display a few examples without printing entire long emails
for name in ['Nazario', 'Nigerian_Fraud', 'Enron', 'SpamAssassin']:
    table = raw[name]
    columns = [c for c in ['subject', 'body', 'label'] if c in table.columns]
    print(f'\n{name}')
    display(table[columns].head(3))

## Step 3 — Standardise and clean the email text
We join the subject and body because both contain useful language. We remove only missing/empty text and exact duplicates. We keep punctuation and wording for the later language model.

In [ ]:
# Cell 5: Create the same simple columns for every source
def prepare_source(table, source_name, wanted_label):
    selected = table.loc[table['label'] == wanted_label].copy()
    subject = selected['subject'].fillna('').astype(str)
    body = selected['body'].fillna('').astype(str)
    selected['text'] = (subject + ' ' + body).str.replace(r'\s+', ' ', regex=True).str.strip()
    selected['source'] = source_name
    selected = selected.loc[selected['text'] != '', ['text', 'label', 'source']]
    before = len(selected)
    selected = selected.drop_duplicates(subset='text').reset_index(drop=True)
    print(f'{source_name}: kept {len(selected):,} emails; removed {before - len(selected):,} duplicates.')
    return selected

nazario_scam = prepare_source(raw['Nazario'], 'Nazario', 1)
nigerian_scam = prepare_source(raw['Nigerian_Fraud'], 'Nigerian_Fraud', 1)
enron_legit = prepare_source(raw['Enron'], 'Enron', 0)
spamassassin_legit = prepare_source(raw['SpamAssassin'], 'SpamAssassin', 0)

## Step 4 — Build balanced, source-separated groups
A balanced dataset has the same number of scam and legitimate examples. This prevents a model from looking accurate merely by choosing the larger class.

In [ ]:
# Cell 6: Training uses Nazario + Enron
RANDOM_SEED = 42
train_per_class = min(len(nazario_scam), len(enron_legit))

train_data = pd.concat([
    nazario_scam.sample(train_per_class, random_state=RANDOM_SEED),
    enron_legit.sample(train_per_class, random_state=RANDOM_SEED),
]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# External testing uses different sources: Nigerian Fraud + SpamAssassin
test_per_class = min(len(nigerian_scam), len(spamassassin_legit))
test_data = pd.concat([
    nigerian_scam.sample(test_per_class, random_state=RANDOM_SEED),
    spamassassin_legit.sample(test_per_class, random_state=RANDOM_SEED),
]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('Training rows:', len(train_data))
print(train_data.groupby(['source', 'label']).size())
print('\nExternal test rows:', len(test_data))
print(test_data.groupby(['source', 'label']).size())

In [ ]:
# Cell 7: Confirm that identical emails do not occur in both groups
overlap = set(train_data['text']) & set(test_data['text'])
print('Exact messages appearing in both training and external testing:', len(overlap))
assert len(overlap) == 0, 'Training/test overlap must be removed before modelling.'

In [ ]:
# Cell 8: Simple chart of the prepared groups
chart_data = pd.DataFrame({
    'Training': train_data['label'].value_counts().sort_index(),
    'External testing': test_data['label'].value_counts().sort_index(),
}).rename(index={0: 'Legitimate', 1: 'Scam'})

chart_data.plot(kind='bar', figsize=(8, 5), color=['#4C78A8', '#E45756'])
plt.title('Prepared training and external test groups')
plt.xlabel('Known label')
plt.ylabel('Number of emails')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
chart_data

## Step 5 — Save the prepared files
These two CSV files will be used by Part 2. They preserve the original text, known label and source.

In [ ]:
# Cell 9: Export prepared data
output_dir = Path('/content/hit401_prepared')
output_dir.mkdir(exist_ok=True)
train_data.to_csv(output_dir / 'training_data.csv', index=False)
test_data.to_csv(output_dir / 'external_test_data.csv', index=False)

archive_path = Path('/content/hit401_prepared_data.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(output_dir / 'training_data.csv', arcname='training_data.csv')
    archive.write(output_dir / 'external_test_data.csv', arcname='external_test_data.csv')

print('Saved:', archive_path)
print('Download it from the Colab Files panel and keep it for Part 2.')

## What to tell the supervisor

> We used public, labelled email corpora. Nazario phishing emails and Enron legitimate emails form the training group. Nigerian Fraud emails and SpamAssassin legitimate emails form an external test group. Because the training and test sources are different, the experiment checks whether the model generalises beyond the sources it learned from. We removed exact duplicates, balanced the two classes and verified that no identical messages appear in both groups.

## Limitation
Dataset labels come from their original sources and may contain errors. Different collection periods and writing styles can also influence the results. Therefore, performance differences cannot be attributed only to scam language.